# NutriEpiDB: A Database For Dietary Compounds With Epigenetic Targets Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the NutriEpiDB dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.sx3s-9110/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("\nLoaded Dataset Metadata:")
print("Name:", metadata['name'])
print("Description:", metadata['description'])
print("Published:", metadata.get('datePublished', 'N/A'))
print("License:", metadata.get('license', 'N/A'))


## 2. Data Overview
NutriEpiDB is a small relational database of six tables: the dietary **compounds**, the human epigenetic **targets** (from EpiFactors), the **binding** affinities of compounds for targets (from BindingDB), the **food sources** of the compounds, the reported epigenetic **mechanisms** of the compounds, and the **articles** they were extracted from.

In [ ]:
import ast
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

record_sets = {rs.name.split('/')[-1].replace('.parquet', ''): rs for rs in dataset.metadata.record_sets}
display(pd.DataFrame([{'Table': name, 'Fields': len(rs.fields), 'Description': rs.description}
                      for name, rs in record_sets.items()])
        .style.hide(axis='index').set_caption('Record sets'))

## 3. Data Extraction
Load the six tables as pandas DataFrames, with the field names as columns (`mlcroissant` names them by field `@id`) and the text decoded (it is returned as bytes). Several fields hold lists stored as text, such as `"['grapes', 'vinegar']"`: they are read as Python lists.

In [ ]:
def decode(value):
    return value.decode('utf-8') if isinstance(value, bytes) else value

def load(name):
    record_set = record_sets[name]
    df = pd.DataFrame(dataset.records(record_set=record_set.id))
    return df.rename(columns={field.id: field.name for field in record_set.fields}).apply(lambda column: column.map(decode))

def as_list(value):
    """A list stored as text ("['a', 'b']") as a list; a missing value as an empty list."""
    if not isinstance(value, str) or value in ('', 'nan'):
        return []
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return [value]
    return parsed if isinstance(parsed, list) else [parsed]

tables = {name: load(name) for name in record_sets}
compounds, targets, binding = tables['compound_data'], tables['target_data'], tables['compound_binding']
foods, mechanisms, articles = tables['food_sources'], tables['mechanism_data'], tables['article_data']
foods['compounds'] = foods['pubchem_cid'].map(as_list)

display(pd.DataFrame({'Rows': {name: len(df) for name, df in tables.items()},
                      'Columns': {name: df.shape[1] for name, df in tables.items()}})
        .style.format('{:,}').set_caption('Loaded tables'))

## 4. Exploratory Data Analysis (EDA)
### 4.1 Overview

In [ ]:
overview = {
    'Dietary compounds (PubChem CIDs)': compounds['pubchem_cid'].nunique(),
    'Epigenetic targets': targets['uniprot_id'].nunique(),
    'Food sources': foods['dietary_source'].nunique(),
    'Compound–mechanism associations': len(mechanisms),
    'Binding measurements': len(binding),
    'Compounds with binding measurements': binding['pubchem_cid'].nunique(),
    'Source articles': len(articles),
    'Articles published': f"{articles['year'].min()}–{articles['year'].max()}",
}
display(pd.DataFrame({'Value': {k: f'{v:,}' if isinstance(v, (int, np.integer)) else v for k, v in overview.items()}})
        .style.set_properties(**{'text-align': 'right'}).set_caption('NutriEpiDB at a glance'))

functions = (targets['function'].replace({'#': 'Not annotated'}).fillna('Not annotated')
             .value_counts().rename('Targets').to_frame())
functions.index.name = 'Function of the epigenetic target'
display(functions.style.set_caption('Epigenetic targets by function'))

Most targets write or erase histone modifications: the database centres on histone acetylation and its enzymes.

### 4.2 Grouping: compounds by epigenetic mechanism
The epigenetic mechanisms reported for the compounds, with the number of compounds and of supporting article references.

In [ ]:
by_mechanism = (mechanisms.groupby('mechanism')
                .agg(Compounds=('pubchem_cid', 'nunique'), References=('number_references', 'sum'))
                .sort_values('Compounds', ascending=False))
by_mechanism.index = by_mechanism.index.str.replace('_', ' ')
by_mechanism.index.name = 'Mechanism'
main_mechanisms = by_mechanism[by_mechanism['Compounds'] >= 10]
print(f'{len(by_mechanism)} mechanism labels; {len(by_mechanism) - len(main_mechanisms)} of them, with fewer than 10 compounds, '
      'are not shown (many are variants of the main ones, such as "histone deacetylases" or "NF-kappaB inhibition").')
display(main_mechanisms.style.format('{:,}').background_gradient(cmap='Blues', subset=['Compounds'])
        .set_caption('Compounds and references by mechanism (at least 10 compounds)'))

top_pairs = (mechanisms.sort_values('number_references', ascending=False).head(10)
             [['name', 'mechanism', 'number_references']]
             .rename(columns={'name': 'Compound', 'mechanism': 'Mechanism', 'number_references': 'References'}))
top_pairs['Mechanism'] = top_pairs['Mechanism'].str.replace('_', ' ')
display(top_pairs.style.hide(axis='index').set_caption('Best-documented compound–mechanism associations'))

The best-documented associations are the methyl-donor nutrients of one-carbon metabolism (folate, methionine, choline, vitamin B12, betaine) and DNA methylation.

### 4.3 Normalizing: binding affinity as pIC50
Most binding measurements are IC50 values (the concentration that halves the target's activity), which span nine orders of magnitude. They are converted to **pIC50** = −log₁₀(IC50 in mol/L): higher is more potent, and 7 corresponds to 100 nM. The binding measurements by target:

In [ ]:
potency = binding[binding['ic50_nm'] > 0].copy()
potency['pIC50'] = 9 - np.log10(potency['ic50_nm'])
print(f"{len(potency):,} of {len(binding):,} binding measurements are IC50 values "
      f"(the others are Ki, Kd or EC50); median pIC50 {potency['pIC50'].median():.2f}.")

by_target = (potency.groupby('target_name')
             .agg(Measurements=('pIC50', 'size'), Compounds=('pubchem_cid', 'nunique'),
                  **{'Median pIC50': ('pIC50', 'median'), 'Best pIC50': ('pIC50', 'max')})
             .sort_values('Measurements', ascending=False))
by_target.index.name = 'Target'
display(by_target.head(10).style.format({'Measurements': '{:,}', 'Median pIC50': '{:.2f}', 'Best pIC50': '{:.2f}'})
        .set_caption('IC50 measurements by target (10 most measured)'))

### 4.4 Filtering: the measured compounds
The binding data is very unevenly spread over the compounds. The compounds with at least three IC50 measurements, by median potency:

In [ ]:
names = compounds.drop_duplicates('pubchem_cid').set_index('pubchem_cid')['name']
by_compound = (potency.groupby('pubchem_cid')
               .agg(Measurements=('pIC50', 'size'), Targets=('uniprot_id', 'nunique'),
                    **{'Median pIC50': ('pIC50', 'median'), 'Best pIC50': ('pIC50', 'max')}))
by_compound.insert(0, 'Compound', names.reindex(by_compound.index).values)
most_measured = by_compound.loc[by_compound['Measurements'].idxmax()]
print(f"{most_measured['Compound']} alone accounts for {most_measured['Measurements'] / len(potency):.0%} of the IC50 measurements.")
display(by_compound[by_compound['Measurements'] >= 3].sort_values('Median pIC50', ascending=False).head(12)
        .style.hide(axis='index')
        .format({'Measurements': '{:,}', 'Median pIC50': '{:.2f}', 'Best pIC50': '{:.2f}'})
        .set_caption('Most potent compounds (at least 3 IC50 measurements)'))

The binding data is dominated by reference histone deacetylase inhibitors (vorinostat, trichostatin A, entinostat), measured on many HDACs, rather than by food compounds: the target-level statistics above mostly describe them.

## 5. Visualization
- **Mechanisms**: the number of compounds reported for each epigenetic mechanism.
- **Articles**: the source articles by year of publication (the last year is incomplete).
- **Potency by target**: the pIC50 of the measurements on the eight most measured targets.
- **Food sources**: the foods with the most compounds (compounds without a food source and generic sources such as "Diet" or "food" are left out).

In [ ]:
BLUE, MUTED = '#2a78d6', '#9ec5f4'
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Mechanisms
ax = axes[0, 0]
top_mechanisms = by_mechanism['Compounds'].head(12)[::-1]
bars = ax.barh(top_mechanisms.index, top_mechanisms.values, color=BLUE, edgecolor='white', linewidth=2)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set(title='Compounds by epigenetic mechanism', xlabel='Compounds')

# Articles by year
ax = axes[0, 1]
per_year = articles['year'].value_counts().sort_index()
last_year = per_year.index.max()
ax.bar(per_year.index, per_year.values, color=[MUTED if year == last_year else BLUE for year in per_year.index],
       edgecolor='white', linewidth=1)
ax.set(title=f'Source articles by year ({last_year}: incomplete)', xlabel='Year', ylabel='Articles')

# Potency by target
ax = axes[1, 0]
top_targets = by_target.head(8).index
data = potency[potency['target_name'].isin(top_targets)]
sns.boxplot(data=data, y='target_name', x='pIC50', order=top_targets, color=MUTED, fliersize=2, ax=ax)
ax.axvline(7, color='#898781', linestyle='--', linewidth=1)
ax.text(7.05, len(top_targets) - 0.6, 'IC50 = 100 nM', fontsize=9, color='#52514e')
ax.set(title='Potency on the eight most measured targets', xlabel='pIC50 (higher is more potent)', ylabel='')

# Food sources
ax = axes[1, 1]
GENERIC = {'diet', 'food', 'plant', 'plants', 'supplement', 'fruit', 'vegetables', 'herb', 'herbs', 'drinks', 'maternal diet'}
specific = foods[foods['dietary_source'].notna() & ~foods['dietary_source'].str.lower().isin(GENERIC)]
top_foods = specific.assign(n=specific['compounds'].map(len)).nlargest(15, 'n').set_index('dietary_source')['n'][::-1]
bars = ax.barh(top_foods.index, top_foods.values, color=BLUE, edgecolor='white', linewidth=2)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set(title='Food sources with the most compounds', xlabel='Compounds')

plt.tight_layout()
plt.show()

no_food = foods.loc[foods['dietary_source'].isna(), 'compounds'].map(len).sum()
print(f'{no_food:,} compounds are listed without a food source.')

## 6. Conclusion
We loaded NutriEpiDB with the `mlcroissant` library and explored its six tables:

- **Targets**: 80 human epigenetic factors, most of them writers or erasers of histone modifications.
- **Mechanisms**: gene expression regulation, epigenetic modulation, histone acetylation, DNA methylation and HDAC inhibition are each reported for 300–400 compounds; the best-documented associations link the methyl donors of one-carbon metabolism (folate, methionine, choline, vitamin B12) to DNA methylation.
- **Binding affinities**: nearly half of the IC50 measurements are below 100 nM, but they cover only about a hundred compounds and are dominated by reference HDAC inhibitors such as vorinostat, mostly on histone deacetylases 1, 2, 3, 6 and 8.
- **Foods**: tea, grapes, turmeric and soy are among the foods with the most compounds, while many compounds have no food source recorded.
- **Literature**: the source articles span 1991 to the present, most of them published since 2015.

Further exploration can link the tables (compounds → foods → mechanisms → targets), restrict the binding data to compounds found in foods, or compare the targets by function.